# ARES: Autonomous Reliable Expert System (Kaggle T4)

This notebook runs the complete 6-stage ARES pipeline + Interactive Web Visualizer:
1. **Setup & Sync Repository**
2. **Multi-Domain Representation Harvesting**
3. **Train Reliability Models (GRM, LRM, Calibration)**
4. **Train Domain PEFT LoRA Experts (Math, Code, Science, Reasoning, General)**
5. **Train Learned Router Policy**
6. **End-to-End ARES Pipeline & Baseline Benchmark Evaluation**
7. **(Optional) Launch Live Interactive Streamlit Web Visualizer**

In [ ]:
%%bash
set -e

echo "=== [1/7] Setting Up Environment & Codebase ==="
if [ -d "ARES-research" ]; then
    cd ARES-research
    git fetch origin
    git reset --hard origin/main
    git pull origin main
else
    git clone https://github.com/sharksurfauto-byte/ARES-research.git
    cd ARES-research
fi

pip install -q --upgrade pip
pip install -q -e .

export TRANSFORMERS_NO_ADVISORY_WARNINGS=1
export TOKENIZERS_PARALLELISM=false
export PYTHONWARNINGS="ignore"

echo "=== Environment Ready ==="

In [ ]:
%%bash
cd /kaggle/working/ARES-research
export TRANSFORMERS_NO_ADVISORY_WARNINGS=1
export TOKENIZERS_PARALLELISM=false

python -W ignore scripts/harvest_real_data.py --model_name "Qwen/Qwen2.5-0.5B" --output_dir "representations/multi_domain" --train_samples_per_domain 400 --val_samples_per_domain 100 --batch_size 4 --max_seq_len 256 --device cuda

In [ ]:
%%bash
cd /kaggle/working/ARES-research
export TRANSFORMERS_NO_ADVISORY_WARNINGS=1

python -W ignore scripts/train_reliability_models.py --data_dir "representations/multi_domain" --output_dir "checkpoints/reliability" --epochs 10 --batch_size 32 --lr 1e-4 --device cuda --calibrate --no_wandb

In [ ]:
%%bash
cd /kaggle/working/ARES-research
export TRANSFORMERS_NO_ADVISORY_WARNINGS=1

python -W ignore scripts/train_experts.py --data_dir "representations/multi_domain" --output_dir "checkpoints/experts" --epochs 8 --batch_size 16 --lr 3e-4 --lora_r 32 --lora_alpha 64 --max_samples 400 --device cuda --no_wandb

In [ ]:
%%bash
cd /kaggle/working/ARES-research
export TRANSFORMERS_NO_ADVISORY_WARNINGS=1

python -W ignore scripts/train_router.py --data_dir "representations/multi_domain" --output_dir "checkpoints/router" --grm_checkpoint "checkpoints/reliability/grm.pt" --lrm_checkpoint "checkpoints/reliability/lrm.pt" --expert_dir "checkpoints/experts" --epochs 10 --batch_size 32 --lr 1e-4 --lambda_lb 0.01 --device cuda --no_wandb

In [ ]:
%%bash
cd /kaggle/working/ARES-research
export TRANSFORMERS_NO_ADVISORY_WARNINGS=1

python -W ignore scripts/run_ares_pipeline.py --model_name "Qwen/Qwen2.5-0.5B" --grm_checkpoint "checkpoints/reliability/grm.pt" --lrm_checkpoint "checkpoints/reliability/lrm.pt" --router_checkpoint "checkpoints/router/router_best.pt" --expert_dir "checkpoints/experts" --benchmark all --samples_per_domain 50 --max_new_tokens 128 --run_baselines --output_report "benchmarks_ares_report.md" --output_json "benchmarks_ares_results.json" --device cuda

echo "=== Summary Results ==="
cat benchmarks_ares_report.md

In [ ]:
# === [7/7] Launch Live Streamlit Web Visualizer Dashboard ===
import subprocess
import time

# Start Streamlit in background
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0", "--server.headless", "true"], cwd="/kaggle/working/ARES-research")
time.sleep(3)

# Expose via localtunnel
import urllib.request
try:
    ip = urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode("utf8").strip()
    print(f"\nTunnel Password: {ip}")
except Exception:
    pass

!npx -y localtunnel --port 8501